### Intuition:
- Considered Algorithms:
  - Linear Kernel
  - Cosine Similarity

* Word to Vector Algorithms:
  * TF-IDF Vectorizer
  * Count Vectorizer

# Importing required libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity

import warnings
warnings.filterwarnings("ignore")

### Loading Dataset

In [ ]:
FILEPATH="../data/train/"

df = pd.read_csv(FILEPATH+"meta_data.csv")

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

## Word to Vector

### TF-IDF Vectorizer

In [ ]:
tf_idf_obj = TfidfVectorizer()
tfvt = tf_idf_obj.fit_transform(df['meta_tags'])

In [ ]:
pd.DataFrame(tfvt.toarray(), columns=tf_idf_obj.get_feature_names_out())

### CountVectorizer

In [ ]:
cv_obj = CountVectorizer(analyzer='word', lowercase=True, stop_words='english')
cvt = cv_obj.fit_transform(df['meta_tags'])

In [ ]:
pd.DataFrame(cvt.toarray(), columns=cv_obj.get_feature_names_out())

### Intuition:
#### We should use TF-IDF Vectorizer with Linear Kernel algorithm because:
- Linear Kernel is faster compared to cosine similarity algorithm but the trade-off is it does not perform L2 regularization. So, as a countermeasure we should use TF-IDF Vectorizer with Linear Kernel because TF-IDF performs L2 regularization.

#### We should use Count Vectorizer with Cosine Similarity algorithm because:
- Cosine similarity inherently perform L2 regularization, as a result it is slower as compared to Linear Kernel. On the other hand, Count Vectorizer do not perform L2 regularization. So, there two are good match for each other.

#### Linear Kernel with TF-IDF Vectorizer is always preferable when the dataset is huge. But in that case before performing TF-IDF vectorization we should perform more accurate meta-data composition to get good similarity scores.

# Method-1: using Linear Kernel with TfidfVectorizer

In [ ]:
similarity_matrix_lk = linear_kernel(tfvt, tfvt)
similarity_matrix_lk

In [ ]:
similarity_matrix_lk[0]

### Testing

In [ ]:
movie_title = "John Carter"

In [ ]:
try:
    print(df.loc[df["title"] == movie_title].index[0])
except IndexError:
    print("Movie not found")

In [ ]:
idx1 = df.loc[df["title"] == movie_title].index[0]

In [ ]:
similarity_scores1 = list(enumerate(similarity_matrix_lk[idx1]))
similarity_scores1

In [ ]:
similarity_scores1 = sorted(similarity_scores1, key=lambda x: x[1], reverse=True)
similarity_scores1

In [ ]:
# top 5 similarities
movies_indices1 = [tpl[0] for tpl in similarity_scores1[1:6]]
scores_list1 = [f"{round(tpl[1],3)*100}%" for tpl in similarity_scores1[1:6]]
print(movies_indices1)
print(scores_list1)
df["title"].iloc[movies_indices1]

# Method-2: using Cosine Similarity with CountVectorizer

In [ ]:
similarity_matrix_cs = cosine_similarity(cvt,cvt)
similarity_matrix_cs

In [ ]:
similarity_matrix_cs[0]

### Testing

In [ ]:
movie_title = "John Carter"
idx2 = df.loc[df["title"] == movie_title].index[0]
similarity_scores2 = list(enumerate(similarity_matrix_cs[idx2]))
similarity_scores2 = sorted(similarity_scores2, key=lambda x: x[1], reverse=True)

In [ ]:
# top 5 similarities
movies_indices2 = [tpl[0] for tpl in similarity_scores2[1:6]]
scores_list2 = [f"{round(tpl[1],3)*100}%" for tpl in similarity_scores2[1:6]]
print(movies_indices2)
print(scores_list2)
df["title"].iloc[movies_indices2]

### Comparison:
- Comparing performance of both algorithms for same input

In [ ]:
def get_similar_movies_lk(movie_title):
    idx = df.loc[df["title"] == movie_title].index[0]
    scores = list(enumerate(similarity_matrix_lk[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    movies_indices = [tpl[0] for tpl in scores[1:6]]
    similarity_scores = [f"{round(tpl[1],3)*100}%" for tpl in scores[1:6]]

    similar_movie_list = list(df["title"].iloc[movies_indices])
    return similar_movie_list,similarity_scores


def get_similar_movies_cs(movie_title):
    idx = df.loc[df["title"] == movie_title].index[0]
    scores = list(enumerate(similarity_matrix_cs[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    movies_indices = [tpl[0] for tpl in scores[1:6]]
    similarity_scores = [f"{round(tpl[1],3)*100}%" for tpl in scores[1:6]]

    similar_movie_list = list(df["title"].iloc[movies_indices])
    return similar_movie_list,similarity_scores

In [ ]:
try:
    movie_title = input("Enter your favorite movie: ").strip()
    print("\nTop 5 matches using Linear Kernel:")
    movies, similar = get_similar_movies_lk(movie_title)
    for i, movie in enumerate(movies):
        print("{} --> with {} similarity".format(movie, similar[i]))

    print("\nTop5 matches using Cosine Similarity:")
    movies, similar = get_similar_movies_cs(movie_title)
    for i, movie in enumerate(movies):
        print("{} --> with {} similarity".format(movie, similar[i]))
except IndexError:
    print("No such movie found in the dataset")

# Cosine similarity is performing better